# Week 8: Robustness Testing — Does It Work with Noisy Telescope Data?
**TMI-102 | PaAC, IIT Roorkee**

---

### What you'll do this week
1. **Rotate** galaxy images (there's no "up" in space!)
2. Add **sensor noise** (simulating a real CCD camera)
3. Add **blur** (simulating Earth's atmosphere)
4. Check if accuracy holds up

### Why this matters
Our clean Kaggle images are perfect. Real telescope data has noise, blur, and cosmic rays. A good model should handle this.

Note: this version uses the **3-class dataset** (elliptical, merger, spiral) and the **Week 5 full fine-tuned model retrained on it** (`w5_full_3class.keras`). Images are perturbed in [0, 1] pixel space and *then* passed through `preprocess_input`, so σ means the same thing as before. The test sample is 100 images per class (300 total), so merger results are not drowned out by the bigger classes.

**Setup for this notebook:** add (1) the Galaxy Zoo competition data and (2) the dataset holding the three `.keras` models as inputs. The next cell rebuilds `Galaxy_dataset_3class` with the same recipe and seed as the retrain notebook, so the test images are the same ones the models were evaluated on.

In [ ]:
# ── Build Galaxy_dataset_3class from the competition files (runs once; skipped if the folder already exists) ──
import os, zipfile
import numpy as np, pandas as pd

COMP = "/kaggle/input/competitions/galaxy-zoo-the-galaxy-challenge"
BUILD_DIR = "/kaggle/working/Galaxy_dataset_3class"
ELL_T, SPI_T, MERGER_T, SEED = 0.80, 0.80, 0.30, 42     # merger = Class8.6 >= 0.30 (change here if mentors say otherwise)

if not os.path.isdir(BUILD_DIR + "/train"):
    from sklearn.model_selection import train_test_split
    df = pd.read_csv(f"{COMP}/training_solutions_rev1.zip", compression="zip")

    # --- elliptical + spiral: EXACTLY the Week 3 recipe (same thresholds, same seed, same split) ---
    df["class"] = np.select([df["Class1.1"] >= ELL_T, df["Class1.2"] >= SPI_T], ["elliptical", "spiral"], default="Tie")
    filtered = df[df["class"].isin(["elliptical", "spiral"])].copy()
    min_cnt = min(filtered["class"].value_counts())
    balanced = pd.concat([
        filtered[filtered["class"] == "spiral"].sample(n=min_cnt, random_state=SEED),
        filtered[filtered["class"] == "elliptical"].sample(n=min_cnt, random_state=SEED)
    ]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    tr, tmp = train_test_split(balanced, test_size=0.30, random_state=SEED, stratify=balanced["class"])
    va, te = train_test_split(tmp, test_size=0.50, random_state=SEED, stratify=tmp["class"])

    # --- merger: simply ADDED as a third class. Galaxies already used as elliptical/spiral are skipped,
    #     so one image never has two labels and the two old classes stay exactly as they were. ---
    mer_all = df[df["Class8.6"] >= MERGER_T]
    mer = mer_all[~mer_all["GalaxyID"].isin(balanced["GalaxyID"])].copy()
    mer["class"] = "merger"
    print(f"merger candidates {len(mer_all)} -> {len(mer)} after skipping {len(mer_all) - len(mer)} already used as elliptical/spiral")
    mtr, mtmp = train_test_split(mer, test_size=0.30, random_state=SEED)
    mva, mte = train_test_split(mtmp, test_size=0.50, random_state=SEED)

    splits = [(tr, "train"), (va, "val"), (te, "test"), (mtr, "train"), (mva, "val"), (mte, "test")]
    for s in ["train", "val", "test"]:
        for c in ["elliptical", "merger", "spiral"]:
            os.makedirs(f"{BUILD_DIR}/{s}/{c}", exist_ok=True)
    target, rows = {}, []
    for d, s in splits:
        for gid, c in zip(d["GalaxyID"], d["class"]):
            target[f"{gid}.jpg"] = (s, c); rows.append((gid, c, s))

    n_ext = 0
    with zipfile.ZipFile(f"{COMP}/images_training_rev1.zip") as z:
        for member in z.namelist():
            base = os.path.basename(member)
            if base in target:
                s, c = target[base]
                with open(f"{BUILD_DIR}/{s}/{c}/{base}", "wb") as f:
                    f.write(z.read(member))
                n_ext += 1
    man = pd.DataFrame(rows, columns=["GalaxyID", "cls", "split"])
    man.to_csv(f"{BUILD_DIR}/manifest.csv", index=False)
    print("extracted", n_ext, "of", len(man))
    print(man.groupby(["split", "cls"]).size().unstack())
    print("Check vs Week 3: elliptical/spiral should be 5692 / 1220 / 1220 each.")
else:
    print("Dataset already built at", BUILD_DIR)

In [ ]:
import glob, os, numpy as np, tensorflow as tf
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array

model = tf.keras.models.load_model(glob.glob('/kaggle/input/**/w5_full_3class.keras', recursive=True)[0])
print(model.input_shape, model.output_shape)

DATA = BUILD_DIR   # built by the cell above
CLASS_NAMES = sorted(os.listdir(f'{DATA}/test'))      # ['elliptical', 'merger', 'spiral'] = Keras class order
print(CLASS_NAMES)

rng = np.random.default_rng(42)
PER_CLASS = 100
paths, labels = [], []
for ci, c in enumerate(CLASS_NAMES):
    files = sorted(os.listdir(f'{DATA}/test/{c}'))
    pick = rng.choice(len(files), PER_CLASS, replace=False)
    paths += [f'{DATA}/test/{c}/{files[i]}' for i in pick]
    labels += [ci] * PER_CLASS

test_images = np.array([img_to_array(load_img(p, target_size=(224, 224))) / 255. for p in paths], dtype='float32')
test_labels = np.array(labels)
print(test_images.shape, test_images.min(), test_images.max(), np.bincount(test_labels))

def predict_classes(model, imgs01):
    """imgs01: (N,224,224,3) in [0,1] -> predicted class indices"""
    return model.predict(preprocess_input(imgs01 * 255.), batch_size=32, verbose=0).argmax(1)

clean_pred = predict_classes(model, test_images)
print("Clean baseline accuracy:", (clean_pred == test_labels).mean())
for ci, c in enumerate(CLASS_NAMES):
    print(f"   {c:>10}: {(clean_pred[test_labels == ci] == ci).mean() * 100:.1f}%")

### Task 1: Visual Demo of Perturbations (10 pts)

Run this cell to see what noise and blur look like on a galaxy image (one per class).

In [ ]:
def show_perturbations(img, title='Galaxy'):
    """Show an image under different perturbation levels."""
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))

    # Row 1: Noise levels
    for i, sigma in enumerate([0, 0.05, 0.10, 0.20]):
        noisy = np.clip(img + np.random.normal(0, sigma, img.shape), 0, 1)
        axes[0, i].imshow(noisy)
        axes[0, i].set_title(f'Noise σ={sigma}')
        axes[0, i].axis('off')

    # Row 2: Blur levels
    for i, blur in enumerate([0, 1.0, 2.0, 3.0]):
        blurred = gaussian_filter(img, sigma=(blur, blur, 0)) if blur > 0 else img
        axes[1, i].imshow(blurred)
        axes[1, i].set_title(f'Blur σ={blur}')
        axes[1, i].axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

for ci, c in enumerate(CLASS_NAMES):
    show_perturbations(test_images[np.where(test_labels == ci)[0][0]], f'Sample {c}')

### Task 2: Noise Robustness Curve (15 pts)

Same test as before, but now accuracy is also tracked **per class**, so you can see which galaxy type breaks first.

In [ ]:
def test_noise(model, images, labels, sigmas=[0, 0.02, 0.05, 0.10, 0.15, 0.20], seed=0):
    rs = np.random.default_rng(seed)
    overall, per_class, pred_dist = [], {c: [] for c in CLASS_NAMES}, []
    for sigma in sigmas:
        noisy = np.clip(images + rs.normal(0, sigma, images.shape), 0, 1).astype('float32')
        pred = predict_classes(model, noisy)
        overall.append((pred == labels).mean())
        for ci, c in enumerate(CLASS_NAMES):
            per_class[c].append((pred[labels == ci] == ci).mean())
        pred_dist.append(np.bincount(pred, minlength=len(CLASS_NAMES)))
        print(f"  Noise σ={sigma:.2f} → Accuracy: {overall[-1] * 100:.1f}%   per class: " +
              "  ".join(f"{c}={per_class[c][-1] * 100:.0f}%" for c in CLASS_NAMES))
    return sigmas, overall, per_class, pred_dist

sigmas, noise_acc, noise_pc, noise_dist = test_noise(model, test_images, test_labels)

plt.figure(figsize=(7, 4.5))
plt.plot(sigmas, noise_acc, 'o-', color='cyan', linewidth=2.5, label='Overall')
for c in CLASS_NAMES:
    plt.plot(sigmas, noise_pc[c], '--', marker='.', label=c)
plt.axhline(0.8, color='red', ls=':', alpha=0.6, label='80% line'); plt.axhline(1/3, color='grey', ls=':', alpha=0.6, label='chance (1/3)')
plt.xlabel('Noise Level (σ)'); plt.ylabel('Accuracy'); plt.ylim(0, 1.02)
plt.title('Robustness to Sensor Noise'); plt.grid(True); plt.legend(); plt.show()

### Task 3: Blur Robustness Curve (15 pts)

Same idea — test accuracy at increasing blur levels.

In [ ]:
def test_blur(model, images, labels, blurs=[0, 0.5, 1.0, 1.5, 2.0, 3.0]):
    overall, per_class, pred_dist = [], {c: [] for c in CLASS_NAMES}, []
    for b in blurs:
        blurred = np.stack([gaussian_filter(im, sigma=(b, b, 0)) for im in images]).astype('float32') if b > 0 else images
        pred = predict_classes(model, blurred)
        overall.append((pred == labels).mean())
        for ci, c in enumerate(CLASS_NAMES):
            per_class[c].append((pred[labels == ci] == ci).mean())
        pred_dist.append(np.bincount(pred, minlength=len(CLASS_NAMES)))
        print(f"  Blur σ={b:.1f} → Accuracy: {overall[-1] * 100:.1f}%   per class: " +
              "  ".join(f"{c}={per_class[c][-1] * 100:.0f}%" for c in CLASS_NAMES))
    return blurs, overall, per_class, pred_dist

blurs, blur_acc, blur_pc, blur_dist = test_blur(model, test_images, test_labels)

plt.figure(figsize=(7, 4.5))
plt.plot(blurs, blur_acc, 's-', color='orange', linewidth=2.5, label='Overall')
for c in CLASS_NAMES:
    plt.plot(blurs, blur_pc[c], '--', marker='.', label=c)
plt.axhline(0.8, color='red', ls=':', alpha=0.6, label='80% line'); plt.axhline(1/3, color='grey', ls=':', alpha=0.6, label='chance (1/3)')
plt.xlabel('Blur σ'); plt.ylabel('Accuracy'); plt.ylim(0, 1.02)
plt.title('Robustness to Atmospheric Seeing'); plt.grid(True); plt.legend(); plt.show()

### Evidence for Q1 and Q2 (just run — it prints the facts your answers need)

In [ ]:
def first_below(levels, accs, thr=0.8):
    for l, a in zip(levels, accs):
        if a < thr:
            return l
    return None

print("Overall accuracy drops below 80% at:")
print("   noise σ =", first_below(sigmas, noise_acc), "| blur σ =", first_below(blurs, blur_acc))
print("\nPer class, first level below 80%:")
for c in CLASS_NAMES:
    print(f"   {c:>10}: noise σ = {first_below(sigmas, noise_pc[c])} | blur σ = {first_below(blurs, blur_pc[c])}")

print("\nWhat does the model predict when it is pushed hardest? (count of predictions per class, out of", len(test_labels), ")")
print("   max noise σ=%.2f :" % sigmas[-1], dict(zip(CLASS_NAMES, noise_dist[-1].tolist())))
print("   max blur  σ=%.1f :" % blurs[-1], dict(zip(CLASS_NAMES, blur_dist[-1].tolist())))
print("\nDrop from clean to worst case:  noise %.1f → %.1f points,  blur %.1f → %.1f points" %
      (noise_acc[0] * 100, noise_acc[-1] * 100, blur_acc[0] * 100, blur_acc[-1] * 100))

### Q1 (5 pts)
Which perturbation hurts accuracy more — noise or blur? Why does that make sense physically?
*(Look at the evidence cell above: which curve falls further, which class falls first, and which class the model collapses into at the highest noise/blur. Think about what each perturbation removes from a merger's tidal tails vs. a spiral's arms vs. a smooth elliptical.)*

> *Write your answer here.*

### Q2 (5 pts)
At what noise/blur level does accuracy drop below 80%? Would a real telescope produce data this bad?
*(With 3 classes, chance is 33%, not 50% — keep that in mind when you describe "close to a coin flip".)*

> *Write your answer here.*

---
**Submit as:** `Week_8_YourName.ipynb`